# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 0          # <-- your full roll number
NAME = ""            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
import re
from collections import Counter

results = {tid: process_ticket(txt) for tid, txt in zip(df["Ticket_ID"], df["Ticket_Text"])}

rows = []
for tid, r in results.items():
    rows.append({"Ticket_ID": tid,
                 "Sentences": len(r["sentences"]),
                 "Tokens": len(r["tokens"]),
                 "Punctuation": len(r["punct"]),
                 "Stopwords": len(r["stops"]),
                 "Unique_Tokens": len(set(r["tokens"])),
                 "Final_Tokens": len(r["final"])})
q1 = pd.DataFrame(rows)
total = q1.drop(columns="Ticket_ID").sum()
total["Ticket_ID"] = "TOTAL"
q1 = pd.concat([q1, total.to_frame().T], ignore_index=True)
print(q1.to_string(index=False))

all_tokens = [t for r in results.values() for t in r["tokens"]]
print(f"\n(TOTAL of Unique_Tokens is the sum of per-ticket values; "
      f"distinct tokens across the whole corpus = {len(set(all_tokens))})")

# Detailed view of one selected ticket (the first focus ticket)
sel = FOCUS[0]
r = results[sel]
print(f"\nSelected ticket {sel}: {df.loc[df.Ticket_ID == sel, 'Ticket_Text'].iloc[0]}")
for key, label in [("tokens", "Tokens"), ("punct", "Punctuation tokens"),
                   ("stops", "Stopwords"), ("final", "Final tokens")]:
    print(f"  {label:20s}: {r[key]}")

# Final tokens that are not ordinary dictionary-style words
final_vocab = sorted({t for r in results.values() for t in r["final"]})
odd = [t for t in final_vocab if not re.fullmatch(r"[a-z]+(?:-[a-z]+)*", t) or t in {"ca", "wo", "sha"}]
print("\nFinal tokens that are not real words:")
for t in odd:
    where = [tid for tid, r in results.items() if t in r["final"]]
    print(f"  {t!r:15} from {where}")

**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:* The list printed above shows them. They come from `word_tokenize()`, which follows Penn Treebank rules and splits clitics off their host word: *can't → ca + n't*, *won't → wo + n't*, *didn't → did + n't*, *Rao's → rao + 's*, *it's → it + 's*. Words like *did*, *is* and *it* are stopwords and get removed, but the fragments `n't`, `'s`, `ca` and `wo` are not in NLTK's stopword list and are not all punctuation, so they make it into the final tokens. Other non-words come from numbers and identifiers that the tokenizer keeps or splits (e.g. `0x80070005`, `2.0.1`, `p.m`, or pieces of emails/URLs such as `//lms.thapar.edu/help`).

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# ---- findall patterns ----
# A "word" must not be part of an email/URL: not preceded by . @ / : and not followed by @ or .letter
B, E = r"(?<![\w'.@/:-])", r"(?![\w@]|\.\w)"
LONG_WORD = r"(?<![\w'.@/:])[A-Za-z]{6,}" + E                             # > 5 letters (also inside Re-installing)
NUMBER    = r"\b0x[0-9A-Fa-f]+\b|(?<![\w.])\d+(?:[.,]\d+)*(?!\w)"         # hex, ints, 95.5, 2.0.1, 1,250
CAPITAL   = B + r"[A-Z][A-Za-z]*(?:[-'][A-Za-z]+)*" + E                   # LMS, Wi-Fi, Rao's
VOWEL     = B + r"[AEIOUaeiou][A-Za-z]*(?:[-'][A-Za-z]+)*" + E            # isn't, e-mail, after

# ---- sub patterns ----
EMAIL = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
URL   = r"(?:https?://|www\.)[^\s,;]+?(?=[.,;!?)]*(?:\s|$))"             # trailing punctuation not included
PHONE = (r"(?:\+\d{1,3}[\s-]?)?(?<!\d)\d{5}[\s-]?\d{5}(?!\d)"             # +91 98765 43210 / +91-98765-43210
         r"|(?<!\d)0\d{2,4}[\s-]\d{3}[\s-]\d{4}(?!\d)")                   # 0175-239-3201

def extract(text):
    return {"long_words": re.findall(LONG_WORD, text),
            "numbers": re.findall(NUMBER, text),
            "capitalized": re.findall(CAPITAL, text),
            "vowel_words": re.findall(VOWEL, text)}

def mask(text):
    text = re.sub(EMAIL, "<EMAIL>", text)
    text = re.sub(URL, "<URL>", text)
    text = re.sub(PHONE, "<PHONE>", text)
    return text

print("=== TEST_LINES ===")
for line in TEST_LINES:
    print("\n" + line)
    for k, v in extract(line).items():
        print(f"  {k:12s}: {v}")
    print(f"  {'masked':12s}: {mask(line)}")

print("\n=== My tickets ===")
q2 = pd.DataFrame([{"Ticket_ID": tid, **extract(txt), "masked": mask(txt)}
                   for tid, txt in zip(df.Ticket_ID, df.Ticket_Text)])
pd.set_option("display.max_colwidth", None)
display(q2)

# Capitalized words at the start of a sentence vs elsewhere
start_caps, other_caps = [], []
for r in results.values():
    for s in r["sentences"]:
        for m in re.finditer(CAPITAL, s):
            (start_caps if s[:m.start()].strip(string.punctuation + " ") == "" else other_caps).append(m.group())
print(f"\nCapitalized words: {len(start_caps) + len(other_caps)} in total")
print(f"  only because they start a sentence: {len(start_caps)} -> {start_caps}")
print(f"  capitalized mid-sentence          : {len(other_caps)} -> {other_caps}")

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:* See the count printed above. Most capitalized words in the tickets are capitalized only because they start a sentence: the generator capitalizes the first character of every ticket, and every ending ("Please help.", "Can you check this?", "It worked yesterday." …) starts with a capital. The ones that are capitalized for their own sake are proper nouns or acronyms such as *LMS, VPN, MATLAB, Python, Wi-Fi, Windows, Dr., Rao* and the pronoun *I*. So a plain "capitalized word" regex is a poor detector of named entities unless sentence-initial words are filtered out.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
from nltk.tokenize import RegexpTokenizer

PATTERN = r"""(?x)
    (?:https?://|www\.)[^\s,;]+?(?=[.,;!?)]*(?:\s|$))      # URLs (without trailing punctuation)
  | [\w.+-]+@[\w-]+(?:\.[\w-]+)+                          # email addresses
  | (?:\+\d{1,3}[\s-]?)?\d{5}[\s-]?\d{5}(?!\d)            # mobile numbers: +91 98765 43210
  | 0\d{2,4}[\s-]\d{3}[\s-]\d{4}(?!\d)                    # landline numbers: 0175-239-3201
  | 0x[0-9A-Fa-f]+                                        # hex codes: 0x80070005
  | \d+(?:[.,]\d+)*                                       # integers, decimals, versions: 95.5, 2.0.1, 1,250
  | (?:[A-Za-z]\.){2,}                                    # abbreviations: p.m.
  | [A-Za-z]+(?:[-'][A-Za-z0-9]+)*                        # words, contractions, hyphenated words
  | \.\.\.                                                # ellipsis
  | [^\w\s]                                               # any other single punctuation mark
"""
_tokenizer = RegexpTokenizer(PATTERN)

def my_tokenize(text):
    return _tokenizer.tokenize(text)

print("=== TEST_LINES ===")
for line in TEST_LINES:
    print("\n" + line)
    print("  my_tokenize  :", my_tokenize(line))
    print("  word_tokenize:", word_tokenize(line))

print("\n=== My tickets ===")
n_diff = 0
for tid, txt in zip(df.Ticket_ID, df.Ticket_Text):
    mt, wt = my_tokenize(txt), word_tokenize(txt)
    if mt != wt:
        n_diff += 1
        print(f"\n{tid}: {txt}")
        print("  my_tokenize  :", mt)
        print("  word_tokenize:", wt)
        print("  only in mine :", [t for t in mt if t not in wt],
              "| only in word_tokenize:", [t for t in wt if t not in mt])
print(f"\n{n_diff} of {len(df)} tickets are tokenized differently.")

**Differences between `my_tokenize` and `word_tokenize` (visible on TEST_LINES):**

1. **Contractions** — `isn't` → word_tokenize gives `['is', "n't"]`, mine keeps `["isn't"]`. word_tokenize uses Penn Treebank rules, which deliberately split clitics so that *n't* can be read as "not"; my pattern lets a word continue across an apostrophe (`[-'][A-Za-z0-9]+`).
2. **Email addresses** — `it.helpdesk@thapar.edu` → word_tokenize gives `['it.helpdesk', '@', 'thapar.edu']` because `@` is treated as a separate symbol; mine keeps the whole address as one token because the email alternative comes before the word/punctuation alternatives.
3. **URLs** — `https://lms.thapar.edu/help` → word_tokenize splits at the colon: `['https', ':', '//lms.thapar.edu/help']`; mine keeps the URL whole and leaves the sentence-final `.` as a separate token.
4. **Phone numbers** — `+91 98765 43210` → word_tokenize gives `['+91', '98765', '43210']` because it splits on whitespace first; my pattern allows spaces/hyphens inside a phone number, so it stays one token.

Both tokenizers keep `state-of-the-art`, `95.5` and `2.0.1` intact, so there's no difference for hyphenated words and decimals/versions.

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:* The count is printed at the end of the cell above. The differences come only from tickets containing contractions (n't, 's), emails, URLs, phone numbers or abbreviations like *p.m.*; plain tickets such as "The VPN keeps disconnecting. Please help." are tokenized the same way by both.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

porter, lancaster, wnl = PorterStemmer(), LancasterStemmer(), WordNetLemmatizer()
vocab = sorted({t for r in results.values() for t in r["final"]})

# Pick the Regexp suffix rule from my own words
ing_words = [w for w in vocab if w.endswith("ing") and len(w) > 5]
ed_words  = [w for w in vocab if w.endswith("ed") and len(w) > 4]
print("Words ending in -ing:", ing_words)
print("Words ending in -ed :", ed_words)
suffixes = [s for s, ws in [("ing", ing_words), ("ed", ed_words)] if len(ws) >= 2] or ["ing"]
REGEXP_RULE = "|".join(f"{s}$" for s in suffixes)
regexp = RegexpStemmer(REGEXP_RULE, min=5)
print("Chosen RegexpStemmer rule:", REGEXP_RULE)

# POS tags taken in context (tag each ticket's token sequence, not isolated words)
pos_of = {}
for r in results.values():
    for w, tag in nltk.pos_tag(r["tokens"]):
        pos_of.setdefault(w, tag)

def to_wordnet_pos(tag):
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

rows = []
for w in vocab:
    tag = pos_of.get(w, "NN")
    out = {"Word": w, "POS": tag,
           "Porter": porter.stem(w), "Lancaster": lancaster.stem(w),
           "Regexp": regexp.stem(w), "Lemma(POS)": wnl.lemmatize(w, to_wordnet_pos(tag))}
    if len({out["Porter"], out["Lancaster"], out["Regexp"], out["Lemma(POS)"]}) > 1:
        rows.append(out)
q4 = pd.DataFrame(rows)
print(f"\n{len(q4)} of {len(vocab)} unique final tokens get different outputs:")
display(q4)

# Help for the Observe question
is_word = lambda s: bool(wordnet.synsets(s))
over = [(r.Word, r.Lancaster, r["Lemma(POS)"]) for _, r in q4.iterrows()
        if len(r.Lancaster) <= len(r["Lemma(POS)"]) - 3]
under = [(r.Word, r.Regexp) for _, r in q4.iterrows()
         if r.Regexp != r.Word and not is_word(r.Regexp)]
print("Over-stemming candidates (word, Lancaster, lemma):", over[:5])
print("Under-stemming candidates (word, Regexp)         :", under[:5])

**Choice of Regexp suffix:** The words printed above show that my tickets have many progressive verb forms ending in **-ing** (from the templates: *disconnecting, opening, running, showing, syncing, asking, uploading, restarting, resetting* …) and past forms ending in **-ed** (*connected, installed, uploaded, changed, locked, logged, stopped* …). So the rule strips `ing$` and/or `ed$` (whichever occurs in at least two of my words), with `min=5` so short words like *need* or *red* aren't damaged.

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:* (Candidates from my own data are printed above.)
- **Over-stemming** (too much removed, so unrelated words merge): Lancaster is the most aggressive. For example, *several → sev* (Porter: *sever*, which collides with the verb "sever"), *university → univers*, *update → upd*, *printer → print* (merges the device with the action).
- **Under-stemming** (forms of one word stay separate): the Regexp stemmer only chops the suffix, so *stopped → stopp* and *running → runn* don't match their base forms *stop* and *run*, and *tried* is left untouched (it ends in *-ied*, not *-ed* after a full stem), so *tried* and *try* stay different.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nltk import bigrams

S1 = [t for r in results.values() for t in r["tokens"]]
S2 = [t for t in S1 if not is_punct(t)]
S3 = [t for r in results.values() for t in r["final"]]
S4 = [porter.stem(t) for t in S3]
SETS = {"S1: all tokens": S1, "S2: no punctuation": S2, "S3: final tokens": S3, "S4: S3 + Porter": S4}

q5 = pd.DataFrame([{"Set": name, "Total tokens": len(s), "Unique tokens": len(set(s)),
                    "TTR": round(len(set(s)) / len(s), 4)} for name, s in SETS.items()])
print(q5.to_string(index=False))

print("\nTop 10 words of S1:", Counter(S1).most_common(10))
print("Top 10 words of S3:", Counter(S3).most_common(10))

freqs = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)
plt.figure(figsize=(6, 4))
plt.plot(np.log(ranks), np.log(freqs), "o-", markersize=4)
plt.xlabel("log(rank)"); plt.ylabel("log(frequency)")
plt.title(f"Zipf plot of S3 (roll {ROLL_NO})")
plt.grid(alpha=0.3); plt.show()

# Bigrams are counted inside each ticket so they never cross ticket boundaries
bi_S1 = Counter(b for r in results.values() for b in bigrams(r["tokens"]))
bi_S3 = Counter(b for r in results.values() for b in bigrams(r["final"]))
print("Top 5 bigrams of S1:", bi_S1.most_common(5))
print("Top 5 bigrams of S3:", bi_S3.most_common(5))

# Stems that merged several distinct S3 words (explains the S3 -> S4 change)
merged = {}
for w in set(S3):
    merged.setdefault(porter.stem(w), set()).add(w)
merged = {k: sorted(v) for k, v in merged.items() if len(v) > 1}
print("\nPorter stems that merged different S3 words:", merged if merged else "none")

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:* TTR = unique tokens / total tokens (values in the table above).
- **S1 → S2:** punctuation is a handful of types (`.`, `?`, `!`, `,`) that repeat in almost every ticket, so dropping it cuts the token count a lot but the type count only a little, and TTR **rises**.
- **S2 → S3:** stopwords (*the, i, is, my, to, it* …) are also few types with high frequency, so removing them pushes TTR **up** again. S3 is mostly content words that repeat less often.
- **S3 → S4:** Porter stemming merges different surface forms into one stem (the merged groups are printed above, e.g. *works/worked/working → work*), so unique tokens go down while the total stays the same, and TTR **falls** a bit. If no words merged in my data, S4's TTR equals S3's.

---
## Q6. Interpretation

In [ ]:
# Data-specific examples for the interpretation answers
ttr = dict(zip(q5.Set, q5.TTR))
print("(a) TTR:", ttr)

# (b) first ticket where the two tokenizers disagree
for tid, txt in zip(df.Ticket_ID, df.Ticket_Text):
    mt, wt = my_tokenize(txt), word_tokenize(txt)
    if mt != wt:
        print(f"(b) {tid}: only mine {[t for t in mt if t not in wt]}  vs  only word_tokenize {[t for t in wt if t not in mt]}")
        break

# (c) Porter stems that are not real words / change meaning
preferred = ["several", "please", "university", "update", "deadline", "slowly", "nobody", "anything", "nothing"]
bad = [(w, porter.stem(w)) for w in preferred if w in vocab and porter.stem(w) != w]
bad += [(w, porter.stem(w)) for w in vocab if porter.stem(w) != w and not wordnet.synsets(porter.stem(w))
        and (w, porter.stem(w)) not in bad]
print("(c) undesirable stems:", bad[:6])

# (d) lemmatizer gives a real word where Porter does not
good = []
for w in vocab:
    lemma = wnl.lemmatize(w, to_wordnet_pos(pos_of.get(w, "NN")))
    stem = porter.stem(w)
    if lemma != stem and wordnet.synsets(lemma) and not wordnet.synsets(stem):
        good.append((w, pos_of.get(w), "Porter:", stem, "Lemma:", lemma))
print("(d) lemma better than stem:", good[:6])

**(a) Why does TTR change from S1 to S4 in your dataset?**
TTR rises from S1 to S2 to S3 because each step removes high-frequency, low-variety tokens: first punctuation (`.`, `?`, `!` repeated in every ticket), then stopwords (*the, i, is, my* …). This lowers the total much more than the number of unique tokens. From S3 to S4 it drops (or stays the same), because Porter stemming merges inflected forms (e.g. *working/worked/works → work*) into one type, so unique tokens fall while total tokens stay the same.

**(b) One example where word_tokenize() and my tokenizer differ:**
`isn't` → word_tokenize: `['is', "n't"]`, my_tokenize: `["isn't"]`. word_tokenize follows Penn Treebank rules and splits the clitic *n't* off so it can be read as "not". My RegexpTokenizer pattern allows an apostrophe inside a word, so the contraction stays one token. (Similarly `it.helpdesk@thapar.edu` becomes `it.helpdesk @ thapar.edu` under word_tokenize but stays one token under mine; see the example printed above for my own tickets.)

**(c) One example where stemming produces an undesirable result:**
Porter turns *several → sever*, which is a different real word ("to cut"), so the meaning changes. *please → pleas* and *university → univers* are not words at all. Lancaster is even more aggressive (*update → upd*). More examples from my data are printed above.

**(d) One example where POS-aware lemmatization preserves meaning better than stemming:**
*tried* (tagged VBD) → Porter gives *tri*, which isn't a word, while the WordNet lemmatizer with `pos='v'` gives **try**, the correct base form. Likewise *crashes* tagged as a verb lemmatizes to *crash*, and adjectives/adverbs keep their real form (*slowly* stays *slowly*, while Porter gives *slowli*). Because the lemmatizer knows the part of speech, it maps a word to a real dictionary form instead of just chopping off a suffix.